# Explore Collider Bias
This notebook reproduces the simulation in `DAG.py`, prints regression results for the full sample and the celebrity subset, and shows side-by-side scatterplots with regression lines. Change the `threshold` variable to see how stronger/weaker selection affects the bias.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression
np.random.seed(42)

In [ ]:
def simulate_and_analyze(n=10000, threshold=1.5, noise_scale=0.5):
    talent = np.random.normal(0,1,size=n)
    beauty = np.random.normal(0,1,size=n)
    celebrity_score = talent + beauty + np.random.normal(0, noise_scale, size=n)
    is_celebrity = (celebrity_score > threshold).astype(int)
    df = pd.DataFrame({'talent': talent, 'beauty': beauty, 'is_celebrity': is_celebrity})
    # Full sample regression
    model_pop = smf.ols('talent ~ beauty', data=df).fit()
    # Conditioned on collider
    df_celebrity = df[df['is_celebrity'] == 1]
    model_collider = smf.ols('talent ~ beauty', data=df_celebrity).fit()
    return df, df_celebrity, model_pop, model_collider

In [ ]:
# Run simulation with default settings
df, df_celebrity, model_pop, model_collider = simulate_and_analyze()
print('Analysis 1: General Population (Unconditioned)')
print(f"Estimated Coefficient for Beauty: {model_pop.params['beauty']:.4f}")
print(f"P-value for Beauty: {model_pop.pvalues['beauty']:.4f}")
print('
Analysis 2: Conditioned on Celebrity Status (Collider)')
print(f"Estimated Coefficient for Beauty: {model_collider.params['beauty']:.4f}")
print(f"P-value for Beauty: {model_collider.pvalues['beauty']:.4f}")

In [ ]:
# Plot full sample vs celebrity subset with linear fit lines
plt.figure(figsize=(12,5))
# Full sample
plt.subplot(1,2,1)
plt.scatter(df['beauty'], df['talent'], alpha=0.2)
coef = np.polyfit(df['beauty'], df['talent'], 1)
xvals = np.linspace(df['beauty'].min(), df['beauty'].max(), 100)
plt.plot(xvals, coef[0]*xvals + coef[1], color='red')
plt.title('Full sample')
plt.xlabel('beauty')
plt.ylabel('talent')
# Celebrity subset
plt.subplot(1,2,2)
plt.scatter(df_celebrity['beauty'], df_celebrity['talent'], alpha=0.3, color='orange')
if len(df_celebrity) > 1:
    coef2 = np.polyfit(df_celebrity['beauty'], df_celebrity['talent'], 1)
    x2 = np.linspace(df_celebrity['beauty'].min(), df_celebrity['beauty'].max(), 100)
    plt.plot(x2, coef2[0]*x2 + coef2[1], color='red')
plt.title('Celebrities only (conditioned on collider)')
plt.xlabel('beauty')
plt.ylabel('talent')
plt.tight_layout()
plt.show()

## Try different thresholds
Change the `threshold` value below and re-run the cell to see how selection strength affects the induced association. Higher thresholds create a smaller selected sample and typically stronger induced bias.

In [ ]:
# Example: re-run with a stronger threshold
threshold = 2.0
df2, df2_celebrity, model_pop2, model_collider2 = simulate_and_analyze(threshold=threshold)
print('Threshold =', threshold)
print('Full sample coef:', f"{model_pop2.params['beauty']:.4f}")
print('Celebrity subset coef:', f"{model_collider2.params['beauty']:.4f}")